# 02_03 - Limpieza base de oferta espacial SER

Este notebook procesa dos fuentes de oferta espacial tabular e infraestructura SER: `ser_calles_plazas` y `ser_parquimetros`. El límite SER y las bandas de aparcamiento se incorporan como dependencias cartográficas limpias generadas por `02_02_cartografia_ser.ipynb`.

El objetivo no es construir todavía oferta agregada, panel SER, joins finales ni métricas proxy, sino dejar las dos fuentes objetivo en `data/interim/ser/...` con un esquema mínimo, validado y trazable. El notebook realiza validaciones espaciales frente al límite SER y compara la capacidad de las bandas cartográficas con la capacidad tabular de `ser_calles_plazas`.

**Entradas y papel dentro del notebook:**

1. `ser_geoportal_limite_ser`: dependencia limpia para controles espaciales.
2. `ser_geoportal_bandas_aparcamiento`: dependencia limpia para comparación por color.
3. `ser_calles_plazas`: capacidad tabular histórica por año, calle/finca/color/plazas.
4. `ser_parquimetros`: infraestructura SER con matrícula, vigencia, calle y coordenadas.

Solo `ser_calles_plazas` y `ser_parquimetros` generan outputs en este notebook. Las dependencias cartográficas se leen como entradas limpias de solo lectura.

## 0. Configuración inicial

Se inicializan rutas, dependencias y constantes. La raíz se detecta automáticamente con `find_repo_root()` a partir de `data_catalog.csv`; cualquier ruta absoluta impresa es solo un diagnóstico local de ejecución, no una dependencia hardcodeada.


In [1]:
from __future__ import annotations

import glob
import re
import unicodedata
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

try:
    import geopandas as gpd
except ImportError as exc:
    raise ImportError(
        "Este notebook requiere GeoPandas y Shapely para las validaciones espaciales. "
        "Instalar con conda install -c conda-forge geopandas pyogrio shapely pyarrow rtree"
    ) from exc

try:
    from shapely.geometry import Point
except ImportError as exc:
    raise ImportError(
        "Este notebook requiere GeoPandas y Shapely para las validaciones espaciales. "
        "Instalar con conda install -c conda-forge geopandas pyogrio shapely pyarrow rtree"
    ) from exc

pd.set_option("display.max_columns", 140)
pd.set_option("display.max_rows", 90)
pd.set_option("display.max_colwidth", 160)

TARGET_DATASET_IDS = [
    "ser_calles_plazas",
    "ser_parquimetros",
]

CARTOGRAPHY_DEPENDENCY_IDS = [
    "ser_geoportal_limite_ser",
    "ser_geoportal_bandas_aparcamiento",
]

REQUIRED_CATALOG_IDS = [
    *TARGET_DATASET_IDS,
    *CARTOGRAPHY_DEPENDENCY_IDS,
]

WINDOW_CALLES = {2023, 2024, 2025, 2026}
SER_WINDOW_START = pd.Timestamp("2023-01-01")
SER_WINDOW_END = pd.Timestamp("2026-12-31")


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current] + list(current.parents):
        if (candidate / "data_catalog.csv").exists():
            return candidate
    raise FileNotFoundError(
        "No se ha encontrado data_catalog.csv. Ejecuta el notebook desde la raiz del repo TFM_parking_madrid."
    )


ROOT = find_repo_root()
CATALOG_PATH = ROOT / "data_catalog.csv"
CHECKLIST_PATH = ROOT / "docs" / "limpieza_SER_checklist.md"
SOURCE_DOCS_ROOT = ROOT / "docs" / "source_docs" / "ser"

print(f"ROOT = {ROOT}")
print(f"CATALOG_PATH exists = {CATALOG_PATH.exists()}")
print(f"CHECKLIST_PATH exists = {CHECKLIST_PATH.exists()}")
print(f"SOURCE_DOCS_ROOT exists = {SOURCE_DOCS_ROOT.exists()}")
print(f"geopandas = {gpd.__version__}")


ROOT = /Users/hugo/TFM_parking_madrid
CATALOG_PATH exists = True
CHECKLIST_PATH exists = True
SOURCE_DOCS_ROOT exists = True
geopandas = 1.1.3


**Lectura/decisión.** Si `ROOT` no apunta a la raíz del repositorio o `geopandas` no carga, el procesamiento no debe continuar. La ruta absoluta impresa ayuda a diagnosticar el entorno local, pero el notebook no depende de esa ruta concreta.


## 1. Catálogo, fuentes objetivo y dependencias cartográficas

Se filtra `data_catalog.csv` por las tres fuentes raw que procesa este notebook y por las dos dependencias cartográficas limpias generadas por `02_02_cartografia_ser.ipynb`.

Para las fuentes objetivo se comprueba que cada patrón de `archivo_raw` resuelve a archivos físicos. Para las dependencias cartográficas se comprueba que el `archivo_interim` catalogado existe y queda disponible como entrada de solo lectura. Los raw cartográficos quedan fuera del alcance de este notebook.

La tabla compacta distingue el rol de cada entrada, la ruta utilizada, su disponibilidad, la salida generada cuando corresponde y el notebook responsable de su producción.

In [2]:
def relpath(path: Path) -> str:
    try:
        return str(path.resolve().relative_to(ROOT))
    except ValueError:
        return str(path)


def resolve_pattern(pattern: str) -> list[Path]:
    raw_pattern = ROOT / pattern
    matches = [Path(p) for p in glob.glob(str(raw_pattern))]
    return sorted(matches)


catalog = pd.read_csv(CATALOG_PATH)
required_catalog_cols = {
    "dataset_id", "bloque", "prioridad", "nombre_fuente", "source_code", "url_fuente",
    "tipo_acceso", "formato", "formato_preferido", "periodo_dato_objetivo",
    "archivo_raw", "archivo_interim", "unidad_espacial", "granularidad_temporal", "estado",
    "notebook_generador",
}
missing_catalog_cols = sorted(required_catalog_cols - set(catalog.columns))
if missing_catalog_cols:
    raise ValueError(f"Faltan columnas obligatorias en data_catalog.csv: {missing_catalog_cols}")

catalog_scope = catalog[catalog["dataset_id"].isin(REQUIRED_CATALOG_IDS)].copy()
missing_dataset_ids = [ds for ds in REQUIRED_CATALOG_IDS if ds not in set(catalog_scope["dataset_id"])]
if missing_dataset_ids:
    raise ValueError(f"Faltan dataset_id en data_catalog.csv: {missing_dataset_ids}")

if catalog_scope["dataset_id"].duplicated().any():
    duplicated_ids = catalog_scope.loc[catalog_scope["dataset_id"].duplicated(), "dataset_id"].tolist()
    raise ValueError(f"Dataset_id duplicados en el alcance del catálogo: {duplicated_ids}")

order = pd.Categorical(catalog_scope["dataset_id"], categories=REQUIRED_CATALOG_IDS, ordered=True)
catalog_scope = catalog_scope.assign(_order=order).sort_values("_order").drop(columns="_order").reset_index(drop=True)
catalog_targets = catalog_scope[catalog_scope["dataset_id"].isin(TARGET_DATASET_IDS)].copy().reset_index(drop=True)
catalog_dependencies = catalog_scope[catalog_scope["dataset_id"].isin(CARTOGRAPHY_DEPENDENCY_IDS)].copy().reset_index(drop=True)

raw_rows = []
RAW_FILES: dict[str, list[Path]] = {}
for row in catalog_targets.itertuples(index=False):
    files = resolve_pattern(row.archivo_raw)
    RAW_FILES[row.dataset_id] = files
    raw_rows.append({
        "dataset_id": row.dataset_id,
        "rol": "fuente_raw_objetivo",
        "archivo_entrada": row.archivo_raw,
        "entrada_existe": bool(files),
        "n_archivos_encontrados": len(files),
        "archivo_salida_generada": row.archivo_interim,
        "notebook_generador": row.notebook_generador,
    })

for row in catalog_dependencies.itertuples(index=False):
    interim_path = ROOT / row.archivo_interim
    raw_rows.append({
        "dataset_id": row.dataset_id,
        "rol": "dependencia_cartografica_limpia",
        "archivo_entrada": row.archivo_interim,
        "entrada_existe": interim_path.exists(),
        "n_archivos_encontrados": 1 if interim_path.exists() else 0,
        "archivo_salida_generada": pd.NA,
        "notebook_generador": row.notebook_generador,
    })

catalog_input_check = pd.DataFrame(raw_rows)
missing_inputs = catalog_input_check.loc[~catalog_input_check["entrada_existe"], ["dataset_id", "archivo_entrada", "rol"]]
if not missing_inputs.empty:
    raise FileNotFoundError(
        "Hay entradas catalogadas no disponibles. "
        f"Detalle: {missing_inputs.to_dict(orient='records')}"
    )

catalog_input_check

,dataset_id,rol,archivo_entrada,entrada_existe,n_archivos_encontrados,archivo_salida_generada,notebook_generador
0,ser_calles_plazas,fuente_raw_objetivo,data/raw/ser/ser_calles_plazas/ser_calles_plazas__*.csv,True,4,data/interim/ser/ser_calles_plazas/ser_calles_plazas_clean.parquet,notebooks/02_03_ser_oferta_espacial.ipynb
1,ser_parquimetros,fuente_raw_objetivo,data/raw/ser/ser_parquimetros/ser_parquimetros__actual.*,True,2,data/interim/ser/ser_parquimetros/ser_parquimetros_clean.parquet,notebooks/02_03_ser_oferta_espacial.ipynb
2,ser_geoportal_limite_ser,dependencia_cartografica_limpia,data/interim/cartografia/ser_geoportal_limite_ser/ser_geoportal_limite_ser_clean.parquet,True,1,NaN,notebooks/02_02_cartografia_ser.ipynb
3,ser_geoportal_bandas_aparcamiento,dependencia_cartografica_limpia,data/interim/cartografia/ser_geoportal_bandas_aparcamiento/ser_geoportal_bandas_aparcamiento_clean.parquet,True,1,NaN,notebooks/02_02_cartografia_ser.ipynb


**Lectura/decisión.** La tabla confirma la disponibilidad de dos fuentes raw objetivo y dos dependencias cartográficas limpias de solo lectura, localizadas en las rutas declaradas por el catálogo.

El patrón de `ser_parquimetros` localiza dos activos físicos, un CSV y un KMZ. La limpieza tabular utiliza exclusivamente el CSV; el KMZ se conserva como formato geográfico alternativo y no interviene en este flujo.

La ausencia de cualquiera de los Parquet cartográficos impide continuar, porque el límite y las bandas son entradas necesarias generadas por `02_02_cartografia_ser.ipynb`.

## 2. Funciones auxiliares tabulares y geoespaciales

Estas funciones auxiliares fijan criterios comunes para todo el notebook antes de limpiar fuentes individuales. Su objetivo es evitar que cada fuente se procese con reglas distintas de lectura, normalización o validación.

Se definen utilidades para normalizar fuentes tabulares, leer CSV/XLSX con separadores y codificaciones habituales, convertir fechas y números, componer códigos de barrio, normalizar colores y nombres de calle, construir puntos, comprobar CRS, cargar y validar dependencias GeoParquet, realizar controles espaciales y escribir Parquet.

Estas funciones no limpian raw cartográfico, no agregan información, no construyen joins finales y no generan métricas proxy. Solo preparan una base técnica común para que las limpiezas posteriores sean reproducibles y comparables entre fuentes.

In [3]:
ENCODINGS = ["utf-8", "utf-8-sig", "latin1", "cp1252"]
SEPARATORS = [";", ",", "\t", "|"]
TABULAR_SUFFIXES = {".csv", ".txt", ".xlsx", ".xls"}


def strip_accents(value: str) -> str:
    return "".join(
        char for char in unicodedata.normalize("NFKD", value)
        if not unicodedata.combining(char)
    )


COLUMN_ALIASES = {
    "gisx": "gis_x",
    "gis_x": "gis_x",
    "gisy": "gis_y",
    "gis_y": "gis_y",
    "cod_distrito": "cod_distrito",
    "codigo_distrito": "cod_distrito",
    "coddis": "cod_distrito",
    "nomdis": "distrito",
    "distrito": "distrito",
    "cod_barrio": "cod_barrio",
    "codigo_barrio": "cod_barrio",
    "cod_distrito_barrio": "cod_barrio",
    "codbar": "num_barrio",
    "num_barrio": "num_barrio",
    "numero_barrio": "num_barrio",
    "nombar": "barrio",
    "barrio": "barrio",
    "calle": "calle",
    "num_finca": "numero_finca",
    "n_finca": "numero_finca",
    "no_finca": "numero_finca",
    "numero_de_finca": "numero_finca",
    "numero_finca": "numero_finca",
    "num_plazas": "numero_plazas",
    "n_plazas": "numero_plazas",
    "no_plazas": "numero_plazas",
    "numero_de_plazas": "numero_plazas",
    "numero_plazas": "numero_plazas",
    "res_numpla": "numero_plazas",
    "res_numplazas": "numero_plazas",
    "color": "color",
    "bateria_linea": "bateria_linea",
    "bateria_li": "bateria_linea",
    "texto_caje": "texto_cajetin",
    "id": "id_banda",
    "objectid": "objectid",
    "nombre": "nombre",
    "codigo_de_via": "codigo_via",
    "codigo_via": "codigo_via",
    "clase_de_la_via": "clase_via",
    "clase_via": "clase_via",
    "particula_de_la_via": "particula_via",
    "particula_via": "particula_via",
    "nombre_de_la_via": "nombre_via",
    "nombre_via": "nombre_via",
    "tipo_de_tramo": "tipo_tramo",
    "tipo_tramo": "tipo_tramo",
    "nombre_de_la_aproximacion": "nombre_aproximacion",
    "nombre_aproximacion": "nombre_aproximacion",
    "numero_inicial_del_tramo": "numero_inicial",
    "numero_inicial": "numero_inicial",
    "calificador_del_numero_inicial_del_tramo": "calificador_numero_inicial",
    "calificador_numero_inicial": "calificador_numero_inicial",
    "numero_final_del_tramo": "numero_final",
    "numero_final": "numero_final",
    "calificador_del_numero_final_del_tramo": "calificador_numero_final",
    "calificador_numero_final": "calificador_numero_final",
    "fecha_de_alta": "fecha_de_alta",
    "fecha_alta": "fecha_de_alta",
    "fecha_de_baja": "fecha_de_baja",
    "fecha_baja": "fecha_de_baja",
    "matricula": "matricula",
    "longitud": "longitud",
    "latitud": "latitud",
}


def normalize_key(value: Any) -> str:
    text = str(value).strip().lower()
    text = strip_accents(text)
    text = re.sub(r"\bn[º°]\b", "numero", text)
    text = text.replace("º", " numero ").replace("°", " numero ")
    text = text.replace("/", "_")
    text = re.sub(r"[^a-z0-9]+", "_", text)
    return re.sub(r"_+", "_", text).strip("_")


def normalize_col(col: Any) -> str:
    key = normalize_key(col)
    return COLUMN_ALIASES.get(key, key)


def make_unique_columns(cols: list[str]) -> list[str]:
    counts: dict[str, int] = {}
    result = []
    for col in cols:
        if col not in counts:
            counts[col] = 0
            result.append(col)
        else:
            counts[col] += 1
            result.append(f"{col}_{counts[col]}")
    return result


def clean_text_series(s: pd.Series) -> pd.Series:
    return (
        s.astype("string")
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA, "<NA>": pd.NA})
    )


def _normalise_decimal_text(value: Any) -> Any:
    if pd.isna(value):
        return pd.NA
    text = str(value).strip().replace("\xa0", "").replace(" ", "")
    if text == "" or text.lower() in {"nan", "none", "<na>"}:
        return pd.NA
    if "," in text:
        text = text.replace(".", "").replace(",", ".")
    return text


def to_numeric_series(s: pd.Series) -> pd.Series:
    # Conserva puntos decimales cuando no hay coma decimal; solo elimina puntos como miles si aparece coma.
    text = clean_text_series(s).map(_normalise_decimal_text)
    return pd.to_numeric(text, errors="coerce")


def clean_identifier_series(s: pd.Series) -> pd.Series:
    return clean_text_series(s).str.replace(r"\.0$", "", regex=True)


def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = make_unique_columns([normalize_col(c) for c in out.columns])
    return out


def read_tabular(path: Path, nrows: int | None = None) -> tuple[pd.DataFrame, dict[str, Any]]:
    # Prueba separadores y codificaciones habituales; escoge la lectura con más columnas detectadas.
    suffix = path.suffix.lower()
    if suffix in {".xlsx", ".xls"}:
        df = pd.read_excel(path, nrows=nrows)
        return df, {"encoding": None, "sep": None, "reader": "read_excel"}
    if suffix not in {".csv", ".txt"}:
        raise ValueError(f"Extension no tabular para lectura base: {path.name}")

    attempts = []
    errors = []
    for encoding in ENCODINGS:
        for sep in SEPARATORS:
            try:
                df = pd.read_csv(path, sep=sep, encoding=encoding, nrows=nrows, low_memory=False)
                attempts.append((df.shape[1], df.shape[0], encoding, sep, df))
            except Exception as exc:
                errors.append({"encoding": encoding, "sep": sep, "error": repr(exc)})
    if not attempts:
        raise ValueError(f"No se pudo leer {path}. Errores iniciales: {errors[:3]}")
    attempts.sort(key=lambda item: (item[0], item[1]), reverse=True)
    n_cols, _, encoding, sep, df = attempts[0]
    return df, {"encoding": encoding, "sep": sep, "reader": "read_csv", "n_cols_detected": n_cols}


def extract_year_from_name(path: Path) -> int | None:
    years = re.findall(r"(20\d{2})", path.name)
    return int(years[-1]) if years else None


def split_code_text(series: pd.Series) -> tuple[pd.Series, pd.Series]:
    text = clean_text_series(series)
    extracted = text.str.extract(r"^\s*([0-9]+)\s*[-\.]?\s*(.*)$")
    code = pd.to_numeric(extracted[0], errors="coerce").astype("Int64")
    name = clean_text_series(extracted[1]).where(extracted[1].notna(), text)
    name = name.mask(name.isna(), text)
    return code, name


def parse_barrio_text(series: pd.Series) -> pd.DataFrame:
    text = clean_text_series(series)
    pattern = text.str.extract(r"^\s*(\d{1,2})\s*[-/]\s*(\d{1,2})\s+(.+)$")
    return pd.DataFrame({
        "cod_distrito_from_barrio": pd.to_numeric(pattern[0], errors="coerce").astype("Int64"),
        "num_barrio_from_barrio": pd.to_numeric(pattern[1], errors="coerce").astype("Int64"),
        "barrio_nombre_from_barrio": clean_text_series(pattern[2]),
    })


def compose_barrio_code(cod_distrito: pd.Series, num_barrio: pd.Series) -> pd.Series:
    cod_distrito_num = pd.to_numeric(cod_distrito, errors="coerce")
    num_barrio_num = pd.to_numeric(num_barrio, errors="coerce")
    return (cod_distrito_num * 100 + num_barrio_num).round().astype("Int64")


def normalize_label(value: Any) -> str | pd.NA:
    if pd.isna(value):
        return pd.NA
    text = strip_accents(str(value)).strip().lower()
    text = re.sub(r"\s+", " ", text)
    return text


SER_COLOR_ALIASES = {
    "043000255 azul": "azul",
    "077214010 verde": "verde",
    "081209246 alta rotacion": "alta rotacion",
    "255000000 rojo": "rojo",
    "255140000 naranja": "naranja",
    "azul": "azul",
    "verde": "verde",
    "alta rotacion": "alta rotacion",
    "rojo": "rojo",
    "naranja": "naranja",
    "gris": "gris",
}


def normalize_ser_color(value: Any) -> str | pd.NA:
    # Unifica colores escritos como texto y como código RGB + texto para comparaciones entre fuentes SER.
    label = normalize_label(value)
    if pd.isna(label):
        return pd.NA
    return SER_COLOR_ALIASES.get(label, label)


def color_for_clean(value: Any) -> str | pd.NA:
    color = normalize_ser_color(value)
    if pd.isna(color):
        return pd.NA
    return str(color).replace(" ", "_")


def normalize_street_name(value: Any) -> str | pd.NA:
    label = normalize_label(value)
    if pd.isna(label):
        return pd.NA
    label = re.sub(r"[^a-z0-9 ]+", " ", label)
    label = re.sub(r"\b(calle|callejon|avenida|avda|paseo|plaza|plz|ronda|glorieta|via)\b", " ", label)
    return re.sub(r"\s+", " ", label).strip()


def union_geometry(gdf: gpd.GeoDataFrame):
    return gdf.geometry.union_all() if hasattr(gdf.geometry, "union_all") else gdf.geometry.unary_union


def ensure_parquet_engine() -> None:
    try:
        import pyarrow  # noqa: F401
    except ImportError as exc:
        raise ImportError("Para escribir Parquet instala pyarrow en el entorno activo.") from exc



def validate_cartography_dependency(
    gdf: gpd.GeoDataFrame,
    dataset_id: str,
    expected_columns: list[str],
    allowed_geometry_types: set[str],
) -> None:
    if not isinstance(gdf, gpd.GeoDataFrame):
        raise TypeError(
            f"Dataset: {dataset_id}; se esperaba GeoDataFrame; observado: {type(gdf).__name__}"
        )
    observed_columns = list(gdf.columns)
    if observed_columns != expected_columns:
        raise ValueError(
            f"Dataset: {dataset_id}; columnas observadas: {observed_columns}; "
            f"columnas esperadas: {expected_columns}"
        )
    if gdf.empty:
        raise ValueError(f"Dataset: {dataset_id}; la dependencia cartográfica está vacía.")
    if gdf.crs is None:
        raise ValueError(f"Dataset: {dataset_id}; CRS no declarado.")
    observed_epsg = gdf.crs.to_epsg()
    if observed_epsg != 25830:
        raise ValueError(
            f"Dataset: {dataset_id}; EPSG observado: {observed_epsg}; esperado: 25830"
        )
    null_geometries = int(gdf.geometry.isna().sum())
    if null_geometries:
        raise ValueError(
            f"Dataset: {dataset_id}; geometrías nulas observadas: {null_geometries}; esperado: 0"
        )
    empty_geometries = int(gdf.geometry.is_empty.sum())
    if empty_geometries:
        raise ValueError(
            f"Dataset: {dataset_id}; geometrías vacías observadas: {empty_geometries}; esperado: 0"
        )
    invalid_geometries = int((~gdf.geometry.is_valid & gdf.geometry.notna()).sum())
    if invalid_geometries:
        raise ValueError(
            f"Dataset: {dataset_id}; geometrías inválidas observadas: {invalid_geometries}; esperado: 0"
        )
    observed_types = set(gdf.geometry.geom_type.dropna().unique())
    unexpected_types = sorted(observed_types - allowed_geometry_types)
    if unexpected_types:
        raise ValueError(
            f"Dataset: {dataset_id}; tipos geométricos observados: {sorted(observed_types)}; "
            f"tipos permitidos: {sorted(allowed_geometry_types)}"
        )

print("Funciones auxiliares cargadas")


Funciones auxiliares cargadas


## 3. Inspección de fuentes objetivo y dependencias cartográficas

Las tres fuentes objetivo se inspeccionan desde raw. Las dos dependencias cartográficas se leen desde `data/interim/cartografia/`, usando las rutas declaradas en `data_catalog.csv`.

No se aplican transformaciones de limpieza a las dependencias cartográficas. Solo se valida que el límite SER y las bandas sean aptos para los controles posteriores: validaciones espaciales frente al límite y comparación por color entre bandas y capacidad tabular.

In [4]:
RAW_TABLES: dict[str, list[dict[str, Any]]] = {}
inspection_rows = []

for dataset_id in TARGET_DATASET_IDS:
    items = []
    for path in RAW_FILES[dataset_id]:
        if path.suffix.lower() not in TABULAR_SUFFIXES:
            continue
        df, meta = read_tabular(path)
        norm = normalize_columns(df)
        items.append({"path": path, "raw": df, "norm": norm, "meta": meta})
    if not items:
        raise ValueError(f"No se cargó ningún archivo tabular para {dataset_id}.")
    RAW_TABLES[dataset_id] = items
    inspection_rows.append({
        "dataset_id": dataset_id,
        "rol": "fuente_raw_objetivo",
        "n_archivos": len(items),
        "shapes": [item["raw"].shape for item in items],
        "columnas": [list(item["norm"].columns) for item in items],
        "epsg": pd.NA,
        "tipos_geometria": pd.NA,
    })


def _dependency_path(dataset_id: str) -> Path:
    rel = catalog_dependencies.loc[
        catalog_dependencies["dataset_id"].eq(dataset_id),
        "archivo_interim",
    ].iloc[0]
    return ROOT / rel


ser_geoportal_limite_ser_clean = gpd.read_parquet(_dependency_path("ser_geoportal_limite_ser"))
validate_cartography_dependency(
    ser_geoportal_limite_ser_clean,
    "ser_geoportal_limite_ser",
    expected_columns=["objectid", "nombre", "geometry"],
    allowed_geometry_types={"Polygon", "MultiPolygon"},
)

ser_geoportal_bandas_aparcamiento_clean = gpd.read_parquet(_dependency_path("ser_geoportal_bandas_aparcamiento"))
validate_cartography_dependency(
    ser_geoportal_bandas_aparcamiento_clean,
    "ser_geoportal_bandas_aparcamiento",
    expected_columns=["id_banda", "color", "numero_plazas", "geometry"],
    allowed_geometry_types={"LineString", "MultiLineString"},
)
if ser_geoportal_bandas_aparcamiento_clean["numero_plazas"].isna().any():
    raise ValueError("Dataset: ser_geoportal_bandas_aparcamiento; numero_plazas contiene nulos.")
allowed_band_colors = {"azul", "verde", "alta_rotacion", "rojo", "naranja"}
observed_band_colors = set(ser_geoportal_bandas_aparcamiento_clean["color"].dropna().astype(str).unique())
unexpected_band_colors = sorted(observed_band_colors - allowed_band_colors)
if unexpected_band_colors:
    raise ValueError(
        "Dataset: ser_geoportal_bandas_aparcamiento; colores no esperados. "
        f"Observado: {unexpected_band_colors}; esperado: {sorted(allowed_band_colors)}"
    )

limite_geom = union_geometry(ser_geoportal_limite_ser_clean)
if limite_geom.is_empty:
    raise ValueError("Dataset: ser_geoportal_limite_ser; la unión del límite está vacía.")
if not limite_geom.is_valid:
    raise ValueError("Dataset: ser_geoportal_limite_ser; la unión del límite no es válida.")
if limite_geom.area <= 0:
    raise ValueError(
        f"Dataset: ser_geoportal_limite_ser; área observada: {limite_geom.area}; esperado: área positiva."
    )

for dataset_id, gdf in {
    "ser_geoportal_limite_ser": ser_geoportal_limite_ser_clean,
    "ser_geoportal_bandas_aparcamiento": ser_geoportal_bandas_aparcamiento_clean,
}.items():
    inspection_rows.append({
        "dataset_id": dataset_id,
        "rol": "dependencia_cartografica_limpia",
        "n_archivos": 1,
        "shapes": [gdf.shape],
        "columnas": list(gdf.columns),
        "epsg": gdf.crs.to_epsg() if gdf.crs is not None else pd.NA,
        "tipos_geometria": sorted(gdf.geometry.geom_type.dropna().unique().tolist()),
    })

inspection_table = pd.DataFrame(inspection_rows)
inspection_table

,dataset_id,rol,n_archivos,shapes,columnas,epsg,tipos_geometria
0,ser_calles_plazas,fuente_raw_objetivo,4,"[(32111, 9), (33700, 12), (34583, 12), (34520, 12)]","[[gis_x, gis_y, distrito, barrio, calle, numero_finca, color, bateria_linea, numero_plazas], [gis_x, gis_y, cod_distrito, distrito, cod_barrio, num_barrio, ...",<NA>,<NA>
1,ser_parquimetros,fuente_raw_objetivo,1,"[(6246, 14)]","[[gis_x, gis_y, fecha_de_alta, fecha_de_baja, cod_distrito, distrito, cod_barrio, num_barrio, barrio, calle, numero_finca, matricula, longitud, latitud]]",<NA>,<NA>
2,ser_geoportal_limite_ser,dependencia_cartografica_limpia,1,"[(1, 3)]","[objectid, nombre, geometry]",25830,[Polygon]
3,ser_geoportal_bandas_aparcamiento,dependencia_cartografica_limpia,1,"[(34450, 4)]","[id_banda, color, numero_plazas, geometry]",25830,[LineString]


**Lectura/decisión.** Las dos fuentes raw objetivo cargan correctamente y muestran sus columnas normalizadas para iniciar las limpiezas específicas. El límite SER y las bandas de aparcamiento cargan desde los outputs catalogados de `02_02_cartografia_ser.ipynb`.

El límite queda disponible como `limite_geom` para las validaciones espaciales de calles/plazas y parquímetros. La capa de bandas queda disponible para la comparación por color con la capacidad tabular de `ser_calles_plazas`. Ambas capas se utilizan exclusivamente como entradas validadas.

## 4. Limpieza de `ser_calles_plazas`

**Qué mide.** `ser_calles_plazas` mide la capacidad tabular histórica de plazas SER por año, coordenadas, calle/finca, color y número de plazas.

**Uso en el TFM.** Es la fuente principal para construir capacidad SER agregable por año, barrio o calle. Se usará en fases posteriores como denominador de oferta tabular, no como geometría lineal de mapa.

**Columnas conservadas.** Se conservan `anio`, `gis_x`, `gis_y`, distrito/barrio, `calle`, `numero_finca`, `color` canónico y `numero_plazas`, porque definen localización, unidad temporal anual, tipo de plaza y capacidad.

**Columnas descartadas.** Se descartan `bateria_linea`, `esquema_documental`, flags y trazabilidad de origen. `esquema_documental` se usa solo para armonizar cambios de esquema entre años; una vez normalizadas las columnas, `anio` conserva la información temporal necesaria.

**Validaciones temporales.** Esta fuente no tiene intervalo `fecha_inicio`/`fecha_fin`: su unidad temporal es el año de publicación extraído del nombre del archivo. Por tanto, la validación temporal correcta es comprobar que el año se parsea, que pertenece a la ventana 2023–2026 y que cada archivo entra en un esquema documental esperado. No procede crear reglas de duración ni solape temporal.

**Validaciones de calidad.** Se comprueban plazas nulas, cero o negativas, color nulo, duplicados exactos, contradicciones de plazas en una misma coordenada/año y posición respecto al límite SER. Los puntos fuera del límite se diagnostican, pero no se eliminan aquí: esa decisión se revisará en notebooks posteriores de joins/mapa.


In [5]:
CALLES_FINAL_COLUMNS = [
    "anio", "gis_x", "gis_y", "cod_distrito", "distrito", "cod_barrio", "num_barrio", "barrio",
    "calle", "numero_finca", "color", "numero_plazas",
]


def diagnose_calles_plazas(items: list[dict[str, Any]]) -> tuple[pd.DataFrame, pd.DataFrame]:
    cleaned_parts = []
    temporal_rows = []

    for item in items:
        df = item["norm"].copy()
        anio = extract_year_from_name(item["path"])
        in_window = anio in WINDOW_CALLES

        if anio is None:
            esquema = pd.NA
            estado_temporal = "anio_no_parseable"
        elif not in_window:
            esquema = pd.NA
            estado_temporal = "anio_fuera_ventana"
        elif anio <= 2023:
            esquema = "hasta_2023"
            estado_temporal = "incluido"
        elif anio == 2024:
            esquema = "2024"
            estado_temporal = "incluido"
        else:
            esquema = "desde_2025"
            estado_temporal = "incluido"

        temporal_rows.append({
            "archivo": relpath(item["path"]),
            "anio_parseado": anio,
            "n_filas_raw": int(len(df)),
            "en_ventana_2023_2026": bool(in_window),
            "esquema_documental": esquema,
            "estado_temporal": estado_temporal,
        })

        if not in_window:
            continue

        for col in [
            "gis_x", "gis_y", "cod_distrito", "distrito", "cod_barrio", "num_barrio", "barrio",
            "calle", "numero_finca", "color", "bateria_linea", "numero_plazas",
        ]:
            if col not in df.columns:
                df[col] = pd.NA

        parsed_cod_distrito, parsed_distrito = split_code_text(df["distrito"])
        cod_distrito_raw = pd.to_numeric(df["cod_distrito"], errors="coerce").astype("Int64")
        cod_distrito = cod_distrito_raw.fillna(parsed_cod_distrito).astype("Int64")
        distrito = parsed_distrito.where(parsed_distrito.notna(), clean_text_series(df["distrito"]))

        barrio_parts = parse_barrio_text(df["barrio"])
        parsed_cod_from_barrio, parsed_barrio_simple = split_code_text(df["barrio"])
        cod_barrio_raw = pd.to_numeric(df["cod_barrio"], errors="coerce")
        num_barrio_raw = pd.to_numeric(df["num_barrio"], errors="coerce")
        num_barrio = (
            num_barrio_raw
            .fillna(barrio_parts["num_barrio_from_barrio"])
            .fillna(cod_barrio_raw.where(cod_barrio_raw <= 99))
            .fillna((cod_barrio_raw % 100).where(cod_barrio_raw > 99))
            .fillna((parsed_cod_from_barrio % 100).where(parsed_cod_from_barrio > 99, parsed_cod_from_barrio))
            .round()
            .astype("Int64")
        )
        cod_barrio = (
            cod_barrio_raw.where(cod_barrio_raw > 99)
            .fillna(parsed_cod_from_barrio.where(parsed_cod_from_barrio > 99))
            .fillna(compose_barrio_code(cod_distrito, num_barrio))
            .round()
            .astype("Int64")
        )
        barrio = (
            barrio_parts["barrio_nombre_from_barrio"]
            .where(barrio_parts["barrio_nombre_from_barrio"].notna(), parsed_barrio_simple)
            .where(lambda s: s.notna(), clean_text_series(df["barrio"]))
        )

        numero_plazas = to_numeric_series(df["numero_plazas"]).round().astype("Int64")
        out = pd.DataFrame({
            "anio": anio,
            "gis_x": to_numeric_series(df["gis_x"]),
            "gis_y": to_numeric_series(df["gis_y"]),
            "cod_distrito": cod_distrito,
            "distrito": clean_text_series(distrito),
            "cod_barrio": cod_barrio,
            "num_barrio": num_barrio,
            "barrio": clean_text_series(barrio),
            "calle": clean_text_series(df["calle"]),
            "numero_finca": clean_text_series(df["numero_finca"]),
            "color": clean_text_series(df["color"]).map(color_for_clean).astype("string"),
            "bateria_linea": clean_text_series(df["bateria_linea"]),
            "numero_plazas": numero_plazas,
            "esquema_documental": esquema,
        })
        out["flag_plazas_nulas"] = out["numero_plazas"].isna()
        out["flag_plazas_cero"] = out["numero_plazas"].fillna(-1).eq(0)
        out["flag_plazas_negativas"] = out["numero_plazas"].fillna(0).lt(0)
        out["flag_coordenadas_nulas"] = out[["gis_x", "gis_y"]].isna().any(axis=1)
        out["flag_numero_finca_sin_asignar"] = out["numero_finca"].str.upper().eq("SIN ASIGNAR").fillna(False)
        out["flag_esquema_anio"] = ~out["anio"].isin(sorted(WINDOW_CALLES))
        cleaned_parts.append(out)

    temporal_check = pd.DataFrame(temporal_rows)
    if not cleaned_parts:
        raise ValueError("No se ha limpiado ningún archivo de ser_calles_plazas en ventana 2023-2026.")
    return pd.concat(cleaned_parts, ignore_index=True), temporal_check


def duplicated_with_different_values(df: pd.DataFrame, keys: list[str], value_col: str) -> pd.DataFrame:
    return (
        df.dropna(subset=keys)
        .groupby(keys, dropna=False)[value_col]
        .nunique(dropna=True)
        .reset_index(name=f"n_{value_col}_distintos")
        .loc[lambda x: x[f"n_{value_col}_distintos"].gt(1)]
    )


ser_calles_plazas_diagnostic, calles_archivos_temporales_check = diagnose_calles_plazas(RAW_TABLES["ser_calles_plazas"])
calles_candidate_clean = ser_calles_plazas_diagnostic.loc[
    ser_calles_plazas_diagnostic["numero_plazas"].notna(),
    CALLES_FINAL_COLUMNS,
].copy()
duplicados_exactos_eliminables_calles = calles_candidate_clean.loc[calles_candidate_clean.duplicated(keep="first")].copy()
n_duplicados_exactos_detectados = int(len(duplicados_exactos_eliminables_calles))
duplicados_exactos_eliminables_por_anio = (
    duplicados_exactos_eliminables_calles
    .groupby("anio", dropna=False)
    .size()
    .astype(int)
    .to_dict()
)

calles_temporal_quality = pd.DataFrame([
    ("n_archivos_raw", int(len(calles_archivos_temporales_check)), "Archivos tabulares detectados para ser_calles_plazas."),
    ("n_archivos_incluidos_2023_2026", int(calles_archivos_temporales_check["en_ventana_2023_2026"].sum()), "Archivos incluidos por año parseado dentro de la ventana 2023-2026."),
    ("n_archivos_anio_no_parseable", int(calles_archivos_temporales_check["estado_temporal"].eq("anio_no_parseable").sum()), "Archivos cuyo año no pudo extraerse del nombre."),
    ("n_archivos_fuera_ventana", int(calles_archivos_temporales_check["estado_temporal"].eq("anio_fuera_ventana").sum()), "Archivos excluidos por estar fuera de la ventana 2023-2026."),
    ("anios_incluidos", sorted(calles_archivos_temporales_check.loc[calles_archivos_temporales_check["en_ventana_2023_2026"], "anio_parseado"].dropna().astype(int).unique().tolist()), "Años efectivamente incluidos en el diagnóstico limpio."),
], columns=["check", "valor", "interpretacion"])

dup_xy_anio = duplicated_with_different_values(
    calles_candidate_clean,
    ["anio", "gis_x", "gis_y"],
    "numero_plazas",
)
dup_xy_anio_color = duplicated_with_different_values(
    calles_candidate_clean,
    ["anio", "gis_x", "gis_y", "color"],
    "numero_plazas",
)

limite_geom_buffer_5m = limite_geom.buffer(5)
gdf_calles = gpd.GeoDataFrame(
    calles_candidate_clean.copy(),
    geometry=[
        Point(xy) if ok else None
        for xy, ok in zip(
            zip(calles_candidate_clean["gis_x"], calles_candidate_clean["gis_y"]),
            calles_candidate_clean["gis_x"].notna() & calles_candidate_clean["gis_y"].notna(),
        )
    ],
    crs="EPSG:25830",
)
calles_sin_coord = gdf_calles.geometry.isna()
calles_dentro = gdf_calles.geometry.within(limite_geom).fillna(False)
calles_dentro_buffer = (
    gdf_calles.geometry
    .within(limite_geom_buffer_5m)
    .fillna(False)
)
calles_limite_check = (
    gdf_calles.assign(
        _sin_coord=calles_sin_coord,
        _fuera_estricto=(~calles_sin_coord & ~calles_dentro),
        _fuera_buffer_5m=(~calles_sin_coord & ~calles_dentro_buffer),
    )
    .groupby("anio", dropna=False)
    .agg(
        n_registros=("anio", "size"),
        n_fuera_limite_estricto=("_fuera_estricto", "sum"),
        n_fuera_limite_buffer_5m=("_fuera_buffer_5m", "sum"),
    )
    .reset_index()
)
calles_limite_check["pct_fuera_limite_estricto"] = (
    calles_limite_check["n_fuera_limite_estricto"] / calles_limite_check["n_registros"] * 100
).round(3)
calles_limite_check["pct_fuera_limite_buffer_5m"] = (
    calles_limite_check["n_fuera_limite_buffer_5m"] / calles_limite_check["n_registros"] * 100
).round(3)

calles_quality = pd.DataFrame([
    ("n_filas_diagnostico", int(len(ser_calles_plazas_diagnostic)), "Registros antes de filtrar plazas nulas."),
    ("n_filas_candidato_clean", int(len(calles_candidate_clean)), "Registros con plazas informadas antes de eliminar duplicados exactos."),
    ("n_plazas_nulas_excluibles", int(ser_calles_plazas_diagnostic["flag_plazas_nulas"].sum()), "Registros sin numero_plazas; no sirven como capacidad."),
    ("n_plazas_cero", int(ser_calles_plazas_diagnostic["flag_plazas_cero"].sum()), "Registros con cero plazas."),
    ("n_plazas_negativas", int(ser_calles_plazas_diagnostic["flag_plazas_negativas"].sum()), "Registros con plazas negativas."),
    ("n_color_nulo", int(calles_candidate_clean["color"].isna().sum()), "Registros candidato clean sin color normalizado."),
    ("n_duplicados_exactos_detectados", n_duplicados_exactos_detectados, "Duplicados exactos en columnas finales; se eliminarán del clean final."),
    ("duplicados_exactos_eliminables_por_anio", duplicados_exactos_eliminables_por_anio, "Año de los registros duplicados exactos que se eliminan; la clave incluye anio."),
    ("n_misma_xy_anio_con_plazas_distintas", int(len(dup_xy_anio)), "Misma coordenada y año con valores distintos de numero_plazas."),
    ("n_misma_xy_anio_color_con_plazas_distintas", int(len(dup_xy_anio_color)), "Misma coordenada, año y color con valores distintos de numero_plazas."),
    ("n_fuera_limite_estricto_total", int(calles_limite_check["n_fuera_limite_estricto"].sum()), "Puntos fuera del límite SER estricto; diagnóstico, no filtrado."),
    ("n_fuera_limite_buffer_5m_total", int(calles_limite_check["n_fuera_limite_buffer_5m"].sum()), "Puntos fuera del límite SER con tolerancia 5 m; diagnóstico, no filtrado."),
], columns=["check", "valor", "interpretacion"])

print("A. Validación temporal de archivos ser_calles_plazas")
display(calles_temporal_quality)
display(calles_archivos_temporales_check)
print("B. Validaciones de calidad de registros ser_calles_plazas")
display(calles_quality)
display(calles_limite_check)


A. Validación temporal de archivos ser_calles_plazas


,check,valor,interpretacion
0,n_archivos_raw,4,Archivos tabulares detectados para ser_calles_plazas.
1,n_archivos_incluidos_2023_2026,4,Archivos incluidos por año parseado dentro de la ventana 2023-2026.
2,n_archivos_anio_no_parseable,0,Archivos cuyo año no pudo extraerse del nombre.
3,n_archivos_fuera_ventana,0,Archivos excluidos por estar fuera de la ventana 2023-2026.
4,anios_incluidos,"[2023, 2024, 2025, 2026]",Años efectivamente incluidos en el diagnóstico limpio.


,archivo,anio_parseado,n_filas_raw,en_ventana_2023_2026,esquema_documental,estado_temporal
0,data/raw/ser/ser_calles_plazas/ser_calles_plazas__2023.csv,2023,32111,True,hasta_2023,incluido
1,data/raw/ser/ser_calles_plazas/ser_calles_plazas__2024.csv,2024,33700,True,2024,incluido
2,data/raw/ser/ser_calles_plazas/ser_calles_plazas__2025.csv,2025,34583,True,desde_2025,incluido
3,data/raw/ser/ser_calles_plazas/ser_calles_plazas__2026.csv,2026,34520,True,desde_2025,incluido


B. Validaciones de calidad de registros ser_calles_plazas


,check,valor,interpretacion
0,n_filas_diagnostico,134914,Registros antes de filtrar plazas nulas.
1,n_filas_candidato_clean,134913,Registros con plazas informadas antes de eliminar duplicados exactos.
2,n_plazas_nulas_excluibles,1,Registros sin numero_plazas; no sirven como capacidad.
3,n_plazas_cero,0,Registros con cero plazas.
4,n_plazas_negativas,0,Registros con plazas negativas.
5,n_color_nulo,0,Registros candidato clean sin color normalizado.
6,n_duplicados_exactos_detectados,4,Duplicados exactos en columnas finales; se eliminarán del clean final.
7,duplicados_exactos_eliminables_por_anio,"{2023: 1, 2024: 1, 2025: 1, 2026: 1}",Año de los registros duplicados exactos que se eliminan; la clave incluye anio.
8,n_misma_xy_anio_con_plazas_distintas,0,Misma coordenada y año con valores distintos de numero_plazas.
9,n_misma_xy_anio_color_con_plazas_distintas,0,"Misma coordenada, año y color con valores distintos de numero_plazas."


,anio,n_registros,n_fuera_limite_estricto,n_fuera_limite_buffer_5m,pct_fuera_limite_estricto,pct_fuera_limite_buffer_5m
0,2023,32110,32,5,0.100,0.016
1,2024,33700,35,6,0.104,0.018
2,2025,34583,38,6,0.110,0.017
3,2026,34520,33,4,0.096,0.012


**Lectura/decisión.** La validación temporal confirma que los cuatro archivos anuales de `ser_calles_plazas` se parsean correctamente desde el nombre del fichero y corresponden a 2023, 2024, 2025 y 2026. No hay archivos con año no parseable ni archivos fuera de la ventana temporal del TFM, por lo que no se excluye información anual de forma silenciosa.

Se excluye cualquier registro con `numero_plazas` nulo, porque no puede utilizarse como capacidad. No hay plazas cero, plazas negativas ni colores nulos.

Los duplicados exactos se calculan sobre las columnas finales, incluyendo `anio`. Por tanto, no se elimina una misma calle por aparecer en años distintos: solo se elimina una repetición idéntica dentro del mismo año, con las mismas coordenadas, barrio, calle, finca, color y plazas. El check `duplicados_exactos_eliminables_por_anio` permite verificar en qué año o años se localizan esos registros eliminables.

No aparecen contradicciones de `numero_plazas` para una misma coordenada y año, ni para una misma coordenada, año y color. Esto es más relevante que contar repeticiones por calle, porque una calle puede tener múltiples fincas o segmentos legítimos.

La validación espacial detecta puntos fuera del límite SER estricto y fuera incluso con buffer de 5 m. No se eliminan en este notebook, porque `ser_calles_plazas` será evaluada de nuevo al construir joins y mapas. Por ahora quedan como incidencia espacial trazada, no como criterio de exclusión.


In [6]:
ser_calles_plazas_clean = calles_candidate_clean.drop_duplicates().reset_index(drop=True)

## 5. Limpieza de `ser_parquimetros`

**Qué mide.** `ser_parquimetros` describe la infraestructura SER de parquímetros, con matrícula, vigencia temporal, calle/finca y coordenadas.

**Uso en el TFM.** Será clave para enlazar tiques con infraestructura mediante `matricula`, aunque este notebook no construye todavía ese join.

**Columnas conservadas.** Se conservan coordenadas (`gis_x`, `gis_y`, `longitud`, `latitud`), fechas de alta/baja, distrito/barrio, `calle`, `numero_finca` y `matricula`, porque definen ubicación, vigencia e identificador operativo.

**Columnas descartadas.** Se descartan flags de diagnóstico y trazabilidad de origen. Los flags se usan solo para diagnosticar y decidir si hay registros sin matrícula, fechas inválidas, bajas fuera de ventana o problemas de vigencia.

**Validaciones temporales.** Se comprueba que las fechas de alta/baja sean parseables, que `fecha_de_baja` nula se trate como parquímetro activo, que no haya vigencias invertidas (`fecha_de_alta > fecha_de_baja`) y que la vigencia interseque la ventana operativa 2023–2026. No se impone una duración máxima: en infraestructura urbana una vigencia larga puede ser perfectamente válida y un umbral arbitrario generaría ruido.

**Validaciones de calidad.** Se comprueban duplicados exactos, registros sin matrícula, coordenadas nulas, reutilización de matrícula, intervalos de vigencia solapados y posición respecto al límite SER. La eliminación operativa se limita a registros que no pueden participar en el pipeline 2023–2026.


In [7]:
PARQUIMETROS_FINAL_COLUMNS = [
    "gis_x", "gis_y", "fecha_de_alta", "fecha_de_baja", "cod_distrito", "distrito",
    "cod_barrio", "num_barrio", "barrio", "calle", "numero_finca", "matricula", "longitud", "latitud",
]
FUTURE_DATE = pd.Timestamp("2099-12-31")


def diagnose_ser_parquimetros(items: list[dict[str, Any]]) -> tuple[pd.DataFrame, int, dict[str, Any]]:
    if len(items) != 1:
        print(f"Aviso: ser_parquimetros tiene {len(items)} archivos tabulares; se usa el primero.")
    df = items[0]["norm"].copy()
    for col in PARQUIMETROS_FINAL_COLUMNS:
        if col not in df.columns:
            df[col] = pd.NA

    cod_distrito = pd.to_numeric(df["cod_distrito"], errors="coerce").astype("Int64")
    cod_barrio_raw = pd.to_numeric(df["cod_barrio"], errors="coerce")
    num_barrio_raw = pd.to_numeric(df["num_barrio"], errors="coerce")
    num_barrio = (
        num_barrio_raw
        .fillna(cod_barrio_raw.where(cod_barrio_raw <= 99))
        .fillna((cod_barrio_raw % 100).where(cod_barrio_raw > 99))
        .round()
        .astype("Int64")
    )
    cod_barrio = (
        cod_barrio_raw.where(cod_barrio_raw > 99)
        .fillna(compose_barrio_code(cod_distrito, num_barrio))
        .round()
        .astype("Int64")
    )
    fecha_alta_raw = clean_text_series(df["fecha_de_alta"])
    fecha_baja_raw = clean_text_series(df["fecha_de_baja"])

    diagnostic = pd.DataFrame({
        "gis_x": to_numeric_series(df["gis_x"]),
        "gis_y": to_numeric_series(df["gis_y"]),
        "fecha_de_alta": pd.to_datetime(fecha_alta_raw, errors="coerce"),
        "fecha_de_baja": pd.to_datetime(fecha_baja_raw, errors="coerce"),
        "cod_distrito": cod_distrito,
        "distrito": clean_text_series(df["distrito"]),
        "cod_barrio": cod_barrio,
        "num_barrio": num_barrio,
        "barrio": clean_text_series(df["barrio"]),
        "calle": clean_text_series(df["calle"]),
        "numero_finca": clean_text_series(df["numero_finca"]),
        "matricula": clean_identifier_series(df["matricula"]),
        "longitud": to_numeric_series(df["longitud"]),
        "latitud": to_numeric_series(df["latitud"]),
    })
    diagnostic["flag_calle_sin_asignar"] = diagnostic["calle"].str.upper().eq("SIN ASIGNAR").fillna(False)
    diagnostic["flag_parquimetro_sin_coordenadas"] = diagnostic[["gis_x", "gis_y"]].isna().any(axis=1)
    diagnostic["flag_fecha_alta_invalida"] = fecha_alta_raw.notna() & diagnostic["fecha_de_alta"].isna()
    diagnostic["flag_fecha_baja_invalida"] = fecha_baja_raw.notna() & diagnostic["fecha_de_baja"].isna()
    diagnostic["flag_fecha_alta_nula"] = diagnostic["fecha_de_alta"].isna()
    diagnostic["flag_fecha_baja_nula"] = diagnostic["fecha_de_baja"].isna()
    diagnostic["flag_parquimetro_dado_baja"] = diagnostic["fecha_de_baja"].notna()
    diagnostic["flag_vigencia_invertida"] = (
        diagnostic["fecha_de_alta"].notna()
        & diagnostic["fecha_de_baja"].notna()
        & diagnostic["fecha_de_alta"].gt(diagnostic["fecha_de_baja"])
    )
    diagnostic["flag_vigencia_fuera_ventana_previa"] = (
        diagnostic["fecha_de_baja"].notna()
        & diagnostic["fecha_de_baja"].lt(SER_WINDOW_START)
    )
    diagnostic["flag_vigencia_fuera_ventana_posterior"] = (
        diagnostic["fecha_de_alta"].notna()
        & diagnostic["fecha_de_alta"].gt(SER_WINDOW_END)
    )
    diagnostic["flag_vigencia_interseca_ventana"] = (
        ~diagnostic["flag_vigencia_fuera_ventana_previa"]
        & ~diagnostic["flag_vigencia_fuera_ventana_posterior"]
    )

    # Duración solo para vigencias cerradas. No se usa como filtro porque una vida útil larga de un parquímetro no es incoherente por sí misma.
    diagnostic["duracion_vigencia_cerrada_dias"] = (
        diagnostic["fecha_de_baja"] - diagnostic["fecha_de_alta"]
    ).dt.days

    duplicados_exactos_eliminables = diagnostic.loc[
        diagnostic.duplicated(subset=PARQUIMETROS_FINAL_COLUMNS, keep="first"),
        PARQUIMETROS_FINAL_COLUMNS,
    ].copy()
    n_before = len(diagnostic)
    diagnostic = diagnostic.drop_duplicates(subset=PARQUIMETROS_FINAL_COLUMNS).reset_index(drop=True)
    n_exact_removed = n_before - len(diagnostic)
    exact_duplicate_summary = {
        "n_registros_eliminables": int(len(duplicados_exactos_eliminables)),
        "matriculas_afectadas": sorted(duplicados_exactos_eliminables["matricula"].dropna().astype(str).unique().tolist()),
        "n_sin_matricula": int(duplicados_exactos_eliminables["matricula"].isna().sum()),
    }
    return diagnostic, n_exact_removed, exact_duplicate_summary


def matricula_interval_overlaps(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    interval_df = df.loc[df["matricula"].notna()].copy()
    interval_df["inicio"] = interval_df["fecha_de_alta"].fillna(SER_WINDOW_START)
    interval_df["fin"] = interval_df["fecha_de_baja"].fillna(FUTURE_DATE)
    for matricula, group in interval_df.sort_values(["matricula", "inicio", "fin"]).groupby("matricula"):
        if len(group) < 2:
            continue
        records = group.reset_index(drop=True)
        for idx in range(len(records) - 1):
            current = records.iloc[idx]
            nxt = records.iloc[idx + 1]
            # Intervalos semiabiertos: si una baja y otra alta coinciden el mismo día, no se cuenta como solape real.
            if current["fin"] > nxt["inicio"]:
                rows.append({
                    "matricula": matricula,
                    "inicio_a": current["inicio"],
                    "fin_a": current["fin"],
                    "inicio_b": nxt["inicio"],
                    "fin_b": nxt["fin"],
                    "gis_x_a": current["gis_x"],
                    "gis_y_a": current["gis_y"],
                    "gis_x_b": nxt["gis_x"],
                    "gis_y_b": nxt["gis_y"],
                })
    return pd.DataFrame(rows)


ser_parquimetros_diagnostic, parquimetros_exact_removed, parquimetros_exact_duplicate_summary = diagnose_ser_parquimetros(RAW_TABLES["ser_parquimetros"])

parquimetros_keep = (
    ser_parquimetros_diagnostic["matricula"].notna()
    & ~ser_parquimetros_diagnostic["flag_fecha_alta_invalida"]
    & ~ser_parquimetros_diagnostic["flag_fecha_baja_invalida"]
    & ~ser_parquimetros_diagnostic["flag_vigencia_invertida"]
    & ser_parquimetros_diagnostic["flag_vigencia_interseca_ventana"]
)
parquimetros_candidate_clean = ser_parquimetros_diagnostic.loc[parquimetros_keep, PARQUIMETROS_FINAL_COLUMNS].copy()

# La reutilización de matrícula relevante para el pipeline se diagnostica sobre el candidato clean,
# no sobre registros históricos excluidos por baja anterior a 2023.
mat_groups = parquimetros_candidate_clean.loc[parquimetros_candidate_clean["matricula"].notna()].groupby("matricula", dropna=False)
duplicados_matricula_resumen = mat_groups.agg(
    n_registros=("matricula", "size"),
    n_coordenadas=("gis_x", lambda s: parquimetros_candidate_clean.loc[s.index, ["gis_x", "gis_y"]].drop_duplicates().shape[0]),
    n_fechas=("fecha_de_alta", lambda s: parquimetros_candidate_clean.loc[s.index, ["fecha_de_alta", "fecha_de_baja"]].drop_duplicates().shape[0]),
).reset_index()
duplicados_matricula_resumen = duplicados_matricula_resumen.loc[duplicados_matricula_resumen["n_registros"].gt(1)]

# Los solapes temporales también se evalúan sobre el candidato clean, ya filtrado por ventana operativa.
matricula_solapes = matricula_interval_overlaps(parquimetros_candidate_clean)
if not matricula_solapes.empty:
    matricula_solapes = matricula_solapes.assign(
        misma_coord=lambda df: (
            df["gis_x_a"].round(3).eq(df["gis_x_b"].round(3))
            & df["gis_y_a"].round(3).eq(df["gis_y_b"].round(3))
        )
    )
n_solapes_misma_coord = int(matricula_solapes["misma_coord"].sum()) if not matricula_solapes.empty else 0
n_solapes_coord_distinta = int((~matricula_solapes["misma_coord"]).sum()) if not matricula_solapes.empty else 0

gdf_parquimetros = gpd.GeoDataFrame(
    parquimetros_candidate_clean.copy(),
    geometry=[
        Point(xy) if ok else None
        for xy, ok in zip(
            zip(parquimetros_candidate_clean["gis_x"], parquimetros_candidate_clean["gis_y"]),
            parquimetros_candidate_clean["gis_x"].notna() & parquimetros_candidate_clean["gis_y"].notna(),
        )
    ],
    crs="EPSG:25830",
)
parq_sin_coord = gdf_parquimetros.geometry.isna()
parq_dentro = (
    gdf_parquimetros.geometry
    .within(limite_geom)
    .fillna(False)
)
parq_dentro_buffer = (
    gdf_parquimetros.geometry
    .within(limite_geom.buffer(5))
    .fillna(False)
)

parquimetros_quality = pd.DataFrame([
    ("n_filas_diagnostico", int(len(ser_parquimetros_diagnostic)), "Registros tras eliminar duplicados exactos evidentes."),
    ("n_filas_candidato_clean", int(len(parquimetros_candidate_clean)), "Registros con matrícula válida y vigencia que interseca la ventana 2023-2026."),
    ("n_duplicados_exactos_eliminados", int(parquimetros_exact_removed), "Duplicados exactos eliminados antes del clean."),
    ("duplicados_exactos_matriculas_afectadas", parquimetros_exact_duplicate_summary["matriculas_afectadas"], "Matrículas afectadas por duplicados exactos eliminados."),
    ("n_sin_matricula_excluibles", int(ser_parquimetros_diagnostic["matricula"].isna().sum()), "Registros sin matrícula; no pueden enlazar con tiques."),
    ("n_baja_antes_2023_excluibles", int(ser_parquimetros_diagnostic["flag_vigencia_fuera_ventana_previa"].sum()), "Parquímetros dados de baja antes de la ventana 2023-2026."),
    ("n_alta_despues_2026_excluibles", int(ser_parquimetros_diagnostic["flag_vigencia_fuera_ventana_posterior"].sum()), "Parquímetros cuya alta empieza después de la ventana 2023-2026."),
    ("n_fecha_alta_invalida", int(ser_parquimetros_diagnostic["flag_fecha_alta_invalida"].sum()), "Textos de fecha_alta no parseables."),
    ("n_fecha_baja_invalida", int(ser_parquimetros_diagnostic["flag_fecha_baja_invalida"].sum()), "Textos de fecha_baja no parseables."),
    ("n_fecha_alta_nula", int(ser_parquimetros_diagnostic["flag_fecha_alta_nula"].sum()), "Registros sin fecha_de_alta; si existieran, se interpretarían como alta previa/desconocida solo para diagnóstico de intervalos."),
    ("n_fecha_baja_nula_activos", int(ser_parquimetros_diagnostic["flag_fecha_baja_nula"].sum()), "Registros sin fecha_de_baja; se interpretan como parquímetros activos."),
    ("n_vigencias_invertidas_excluibles", int(ser_parquimetros_diagnostic["flag_vigencia_invertida"].sum()), "Registros con fecha_de_alta posterior a fecha_de_baja."),
    ("n_sin_coordenadas", int(ser_parquimetros_diagnostic["flag_parquimetro_sin_coordenadas"].sum()), "Registros sin coordenadas."),
    ("n_matriculas_reutilizadas", int(len(duplicados_matricula_resumen)), "Matrículas con más de un registro dentro del candidato clean."),
    ("n_matriculas_con_fechas_distintas", int((duplicados_matricula_resumen["n_fechas"].gt(1)).sum()), "Matrículas repetidas con fechas de vigencia distintas dentro del candidato clean."),
    ("n_matriculas_con_coordenadas_distintas", int((duplicados_matricula_resumen["n_coordenadas"].gt(1)).sum()), "Matrículas repetidas con coordenadas distintas dentro del candidato clean."),
    ("n_matriculas_con_intervalos_solapados", int(matricula_solapes["matricula"].nunique()) if not matricula_solapes.empty else 0, "Matrículas con intervalos de vigencia solapados usando criterio estricto dentro del candidato clean."),
    ("n_solapes_misma_coord", n_solapes_misma_coord, "Solapes temporales cuya coordenada coincide redondeando a 1 mm."),
    ("n_solapes_coord_distinta", n_solapes_coord_distinta, "Solapes temporales con coordenadas distintas."),
    ("n_fuera_limite_estricto", int((~parq_sin_coord & ~parq_dentro).sum()), "Parquímetros candidato clean fuera del límite SER estricto."),
    ("n_fuera_limite_buffer_5m", int((~parq_sin_coord & ~parq_dentro_buffer).sum()), "Parquímetros candidato clean fuera del límite SER con tolerancia 5 m."),
], columns=["check", "valor", "interpretacion"])

display(parquimetros_quality)

if not matricula_solapes.empty:
    solapes_cols = [
        "matricula",
        "inicio_a", "fin_a",
        "inicio_b", "fin_b",
        "gis_x_a", "gis_y_a",
        "gis_x_b", "gis_y_b",
        "misma_coord",
    ]
    display(matricula_solapes[solapes_cols].head(5))


,check,valor,interpretacion
0,n_filas_diagnostico,6245,Registros tras eliminar duplicados exactos evidentes.
1,n_filas_candidato_clean,4772,Registros con matrícula válida y vigencia que interseca la ventana 2023-2026.
2,n_duplicados_exactos_eliminados,1,Duplicados exactos eliminados antes del clean.
3,duplicados_exactos_matriculas_afectadas,[],Matrículas afectadas por duplicados exactos eliminados.
4,n_sin_matricula_excluibles,27,Registros sin matrícula; no pueden enlazar con tiques.
5,n_baja_antes_2023_excluibles,1450,Parquímetros dados de baja antes de la ventana 2023-2026.
6,n_alta_despues_2026_excluibles,0,Parquímetros cuya alta empieza después de la ventana 2023-2026.
7,n_fecha_alta_invalida,0,Textos de fecha_alta no parseables.
8,n_fecha_baja_invalida,0,Textos de fecha_baja no parseables.
9,n_fecha_alta_nula,0,"Registros sin fecha_de_alta; si existieran, se interpretarían como alta previa/desconocida solo para diagnóstico de intervalos."


**Lectura/decisión.** La validación temporal de `ser_parquimetros` confirma que las fechas de alta y baja son parseables, que no existen vigencias invertidas y que no hay registros cuya alta comience después de 2026. Las bajas nulas se interpretan como parquímetros activos, lo que afecta a 4679 registros. El filtro conserva 4772 parquímetros con matrícula válida y vigencia que intersecta la ventana 2023–2026; se excluyen 27 registros sin matrícula, 1450 dados de baja antes de 2023 y se elimina el duplicado exacto.

No se aplica una regla de duración máxima. En esta fuente, una vigencia larga no es necesariamente incoherente: un parquímetro puede permanecer activo muchos años. Por eso la duración cerrada queda solo como variable de diagnóstico interno y no como criterio de filtrado.

Los duplicados exactos se eliminan solo si coinciden todas las columnas finales: coordenadas, fechas, distrito/barrio, calle/finca, matrícula, longitud y latitud. Por tanto, no se eliminan reutilizaciones históricas de matrícula con fechas o coordenadas distintas.

Las reutilizaciones de matrícula se diagnostican sobre el candidato clean, ya filtrado por matrícula y vigencia compatible con 2023–2026. Si aparece alguna matrícula con intervalo de vigencia solapado usando criterio estricto, queda trazada para el notebook de joins con tiques, donde se decidirá usando la fecha real de cada tique.

Respecto al límite SER, los parquímetros fuera del límite estricto o incluso fuera con buffer de 5 m no se eliminan en este notebook, porque `ser_parquimetros` será evaluada de nuevo al construir joins y mapas. Por ahora quedan como incidencia espacial trazada, no como criterio de exclusión.


In [8]:
ser_parquimetros_clean = parquimetros_candidate_clean.copy()

## 6. Validaciones cruzadas ligeras entre fuentes limpias

Esta sección compara fuentes ya limpias sin construir joins finales. El objetivo es detectar incoherencias tempranas entre la capa cartográfica que se usará para mapas y las fuentes tabulares que se usarán para capacidad o enlace con tiques.

Se realizan dos validaciones ligeras: comparación de plazas por color entre bandas Geoportal y `ser_calles_plazas` 2025, y cobertura aproximada de nombres de calle de parquímetros dentro de `ser_calles_plazas`. Estas validaciones no eliminan registros ni corrigen geometrías.


In [9]:
geo_color = (
    ser_geoportal_bandas_aparcamiento_clean
    .assign(color_norm=lambda df: df["color"].str.replace("_", " ", regex=False))
    .groupby("color_norm", dropna=False)["numero_plazas"]
    .sum(min_count=1)
    .reset_index(name="plazas_geoportal")
)
calles_2025_color = (
    ser_calles_plazas_clean
    .loc[lambda df: df["anio"].eq(2025)]
    .assign(color_norm=lambda df: df["color"].str.replace("_", " ", regex=False))
    .groupby("color_norm", dropna=False)["numero_plazas"]
    .sum(min_count=1)
    .reset_index(name="plazas_calles_2025")
)
comparacion_plazas_color = geo_color.merge(calles_2025_color, on="color_norm", how="outer")
comparacion_plazas_color[["plazas_geoportal", "plazas_calles_2025"]] = comparacion_plazas_color[["plazas_geoportal", "plazas_calles_2025"]].fillna(0)
comparacion_plazas_color["diferencia_abs"] = comparacion_plazas_color["plazas_geoportal"] - comparacion_plazas_color["plazas_calles_2025"]
comparacion_plazas_color["diferencia_pct"] = np.where(
    comparacion_plazas_color["plazas_calles_2025"].ne(0),
    comparacion_plazas_color["diferencia_abs"] / comparacion_plazas_color["plazas_calles_2025"] * 100,
    np.nan,
)
comparacion_plazas_color["diferencia_pct"] = comparacion_plazas_color["diferencia_pct"].round(2)
comparacion_plazas_color = comparacion_plazas_color.sort_values("color_norm", na_position="last")

calles_parq_norm = set(ser_parquimetros_clean["calle"].map(normalize_street_name).dropna())
calles_cap_norm = set(ser_calles_plazas_clean["calle"].map(normalize_street_name).dropna())
calles_parquimetros_no_en_calles = sorted(calles_parq_norm - calles_cap_norm)
calles_nombre_check = pd.DataFrame([{
    "n_calles_parquimetros": int(len(calles_parq_norm)),
    "n_calles_parquimetros_en_calles_plazas": int(len(calles_parq_norm & calles_cap_norm)),
    "pct_calles_parquimetros_en_calles_plazas": round(len(calles_parq_norm & calles_cap_norm) / len(calles_parq_norm) * 100, 3) if calles_parq_norm else np.nan,
    "ejemplos_no_encontrados": calles_parquimetros_no_en_calles[:20],
}])

print("D. Comparación de plazas por color: Geoportal reguladas vs calles/plazas 2025")
display(comparacion_plazas_color)
print("E. Validación ligera de nombres de calle parquímetros vs calles/plazas")
display(calles_nombre_check)


D. Comparación de plazas por color: Geoportal reguladas vs calles/plazas 2025


,color_norm,plazas_geoportal,plazas_calles_2025,diferencia_abs,diferencia_pct
0,alta rotacion,372,372,0,0.00
1,azul,21183,21307,-124,-0.58
2,naranja,1464,1464,0,0.00
3,rojo,342,358,-16,-4.47
4,verde,157644,157888,-244,-0.15


E. Validación ligera de nombres de calle parquímetros vs calles/plazas


,n_calles_parquimetros,n_calles_parquimetros_en_calles_plazas,pct_calles_parquimetros_en_calles_plazas,ejemplos_no_encontrados
0,1791,1756,98.046,"[alabastro del, bejar de, benito valderas de, caoba de la, caolin del, carmen del, cenicero de, circon del, circonita de la, cordon del, cuarzo del, el esco..."


**Lectura/decisión.** La comparación por color muestra coherencia alta entre la capa lineal Geoportal y `ser_calles_plazas` 2025. Alta rotación y naranja coinciden exactamente; azul difiere en -124 plazas (-0,58 %), verde en -244 plazas (-0,15 %) y rojo en -16 plazas (-4,47 %). En términos absolutos y relativos, la discrepancia global es pequeña y compatible con diferencias de actualización, geometría o codificación entre fuentes.

La validación de nombres de calle muestra que el 98,046 % de las calles normalizadas de parquímetros aparece también en `ser_calles_plazas`. Esta cobertura es suficiente para continuar con futuras pruebas de join por calle, aunque los ejemplos no encontrados deberán revisarse cuando se construya el notebook específico de joins.

La decisión es mantener ambas validaciones como evidencia de compatibilidad inicial, sin crear todavía joins finales ni eliminar registros.


## 7. Escritura de salidas limpias

Se escriben dos salidas limpias en `data/interim/ser/...`: `ser_calles_plazas` y `ser_parquimetros`. No se escribe ninguna salida en `data/processed`, ni se generan agregados, paneles ni métricas proxy.

El límite SER y las bandas de aparcamiento actúan como dependencias limpias de solo lectura generadas por `02_02_cartografia_ser.ipynb`. Este notebook no genera salidas cartográficas.

In [10]:
ensure_parquet_engine()

clean_outputs = {
    "ser_calles_plazas": ser_calles_plazas_clean,
    "ser_parquimetros": ser_parquimetros_clean,
}

expected_output_names = {
    "ser_calles_plazas_clean.parquet",
    "ser_parquimetros_clean.parquet",
}
actual_output_names = {Path(catalog_targets.loc[catalog_targets["dataset_id"].eq(ds), "archivo_interim"].iloc[0]).name for ds in clean_outputs}
if actual_output_names != expected_output_names:
    raise ValueError(f"Las salidas catalogadas no coinciden exactamente: {actual_output_names}")

write_rows = []
for dataset_id in TARGET_DATASET_IDS:
    df = clean_outputs[dataset_id]
    if df.empty:
        raise ValueError(f"La salida limpia de {dataset_id} esta vacia; no se escribe.")
    out_rel = catalog_targets.loc[catalog_targets["dataset_id"].eq(dataset_id), "archivo_interim"].iloc[0]
    out_path = ROOT / out_rel
    out_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(out_path, index=False)
    write_rows.append({
        "dataset_id": dataset_id,
        "archivo_interim": relpath(out_path),
        "shape_escrita": df.shape,
        "size_mb": round(out_path.stat().st_size / 1024**2, 3),
        "estado_escritura": "OK",
    })

write_check = pd.DataFrame(write_rows)
display(write_check)

,dataset_id,archivo_interim,shape_escrita,size_mb,estado_escritura
0,ser_calles_plazas,data/interim/ser/ser_calles_plazas/ser_calles_plazas_clean.parquet,"(134909, 12)",1.360,OK
1,ser_parquimetros,data/interim/ser/ser_parquimetros/ser_parquimetros_clean.parquet,"(4772, 14)",0.239,OK


**Lectura/decisión.** La escritura confirma la generación de dos Parquet limpios, todos con estado `OK`: `ser_calles_plazas` y `ser_parquimetros`.

Las dimensiones escritas son `(134909, 12)` para `ser_calles_plazas` y `(4772, 14)` para `ser_parquimetros`. El límite SER y las bandas de aparcamiento han actuado como dependencias de solo lectura. No se generan outputs cartográficos.

No aparecen salidas agregadas ni processed, por lo que el notebook se mantiene dentro del alcance de limpieza individual de oferta espacial tabular e infraestructura.